In [757]:
def find_k_nearest_users(matrix, target_user, k, similarity_fn):
    users=matrix.index.tolist()
    k_nearest_users = {}
    for user in users:
        if user != target_user:
            user_a,user_b=get_co_rated(matrix, target_user, user, axis="index")
            if(len(user_a)>=2):
                similarity= float(similarity_fn(user_a, user_b))
                if np.isnan(similarity) or similarity <= 0:   
                    continue
                k_nearest_users[user] = similarity

    k_nearest_users = sorted(k_nearest_users.items(), key=lambda x: x[1], reverse=True)
    return k_nearest_users[:k]

In [758]:
def predict_rating_user_based(matrix, target_user, target_item, k, similarity_fn) :
    k_nearest_users = find_k_nearest_users(matrix, target_user, k, similarity_fn)
    numerator = 0
    denominator = 0
    for user in k_nearest_users:
        user_id=user[0]
        rating = matrix.loc[user_id, target_item]
        if pd.notna(rating):
            similarity = user[1]
            numerator += similarity * rating
            denominator += abs(similarity)
    if denominator == 0:
        return np.nan
    return numerator/denominator



In [759]:
def find_k_nearest_items(matrix, target_item, k, similarity_fn):
    movies=matrix.columns.tolist()
    k_nearest_movies={}
    for movie in movies:
        if movie!=target_item:
            item_a,item_b=get_co_rated(matrix, target_item, movie, axis="columns")
            if(len(item_a)>=2):
                similarity= float(similarity_fn(item_a, item_b))
                if np.isnan(similarity) or similarity <= 0:   # drop NaN and negative
                    continue
                k_nearest_movies[movie] = similarity
    k_nearest_movies = sorted(k_nearest_movies.items(), key=lambda x: x[1], reverse=True)
    return k_nearest_movies[:k]

In [760]:
def predict_rating_item_based(matrix, target_user, target_item, k, similarity_fn):
    k_nearest_items= find_k_nearest_items(matrix, target_item, k, similarity_fn)
    numerator=0
    denominator=0
    for item in k_nearest_items:
        item_id=item[0]
        rating=matrix.loc[target_user,item_id]
        if pd.notna(rating):
            similarity = item[1]
            numerator += similarity * rating
            denominator += abs(similarity)
    if denominator == 0:
        return np.nan
    return numerator/denominator



I reused both methods from part C, but instead of searching the rows(users), I searched the columns (movies) to get the nearest movies to the target movie, also the prediction was made on the nearest movies that the user rated.

In [761]:
def mean_centering(matrix,target_user,target_item,k,similarity_fn):
    user_means=matrix.mean(axis=1) #get the mean of all the rows
    centered_matrix = matrix.sub(user_means,axis=0) #subtract the mean from each row.
    rating=predict_rating_user_based(centered_matrix,target_user,target_item,k,similarity_fn)
    mean_user_rating=np.nanmean(np.asarray(matrix.loc[target_user]))
    return np.clip(rating + mean_user_rating, 0.5, 5)
        
        

In [762]:
def calculate_RMSE(actual,predicted):
    actual_vec=np.asarray(actual)
    predicted_vec=np.asarray(predicted)
    return np.sqrt(np.nanmean((actual - predicted) ** 2))

In [763]:
def get_predicted_matrix(matrix,prediction_fn,k,similarity_fn):
    users=matrix.index.tolist()
    movies=matrix.columns.tolist()
    predicted_ratings=pd.DataFrame(np.nan,index=matrix.index,columns=matrix.columns)
    matrix_copy=matrix.copy()
    
    for user in users:
        for movie in movies:
            true_rating=matrix.loc[user,movie]
            if pd.isna(matrix.loc[user, movie]):
                continue
            matrix_copy.loc[user,movie]=np.nan
            prediction=prediction_fn(matrix_copy,user,movie,k,similarity_fn)
            matrix_copy.loc[user,movie]=true_rating
            predicted_ratings.loc[user,movie]=prediction
            
    return predicted_ratings
